# Retrieval-Gated VLM Inference — full pipeline

Linear, reproducible version of the experiments behind
*Lightweight, Provenance-Aware Retrieval Gating for Vision-Language Inference on
Resource-Constrained Devices*.

Original development happened across several sessions on a Kaggle T4; this notebook
reorders those sessions into the order a reader should run them. Every artifact is
written to `artifacts/`, and the committed copies live in `results/`.

**Hardware assumed:** one NVIDIA T4 (16 GB), FAISS on CPU.
**Runtime:** roughly 40-60 minutes end to end, dominated by Step 6 (INT8 is slow by design).


## 0. Setup

Run this cell first in every session. After the first install, restart the runtime once and re-run it — `num2words` availability is cached by `transformers` at import time.


In [ ]:
import os
os.makedirs('artifacts', exist_ok=True)


In [ ]:
# =====================================================================
# CONSOLIDATED SETUP CELL — run this first, every session, before anything else
# Covers all fixes found through Day 2: num2words, dataset revision, CLIP embedding bug
# =====================================================================

# --- Install dependencies ---
!pip install -q -U transformers>=4.50 accelerate faiss-cpu datasets pillow num2words

# --- IMPORTANT: after first-time install, do Kaggle -> Run -> Restart Session,
# then re-run this whole cell once more before continuing. This is required
# specifically for num2words (transformers caches its availability at import
# time, so installing mid-session doesn't take effect until restart).

import torch
import faiss
import numpy as np
import time
from transformers import CLIPModel, CLIPProcessor, AutoProcessor, AutoModelForImageTextToText
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Load Dataset A (Flickr30k) — revision= bypasses the unsupported loading script ---
N_SAMPLES = 2000
dataset_a = load_dataset(
    "nlphuji/flickr30k",
    split=f"test[:{N_SAMPLES}]",
    revision="refs/convert/parquet",
)
images_a = [dataset_a[i]["image"].convert("RGB") for i in range(len(dataset_a))]
print(f"Loaded {len(images_a)} images from Dataset A")

# --- Load CLIP (frozen) ---
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

# --- CLIP embedding functions — FIXED to bypass the broken get_image_features/
# get_text_features wrappers (they return BaseModelOutputWithPooling instead of
# the projected embedding in this transformers version) ---

@torch.no_grad()
def embed_images(images, batch_size=64):
    """images: list of PIL Images -> np.ndarray [N, D] L2-normalized"""
    all_embeds = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i + batch_size]
        inputs = clip_processor(images=batch, return_tensors="pt").to(device)
        vision_outputs = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        pooled_output = vision_outputs.pooler_output
        feats = clip_model.visual_projection(pooled_output)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_embeds.append(feats.cpu().numpy())
    return np.concatenate(all_embeds, axis=0)

@torch.no_grad()
def embed_text(query):
    """Single text query -> np.ndarray [1, D] L2-normalized"""
    inputs = clip_processor(text=[query], return_tensors="pt", padding=True).to(device)
    text_outputs = clip_model.text_model(**inputs)
    pooled_output = text_outputs.pooler_output
    feats = clip_model.text_projection(pooled_output)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()

# Alias — some Day 2+ scripts call this embed_text_query instead of embed_text
embed_text_query = embed_text

@torch.no_grad()
def embed_texts(texts, batch_size=64):
    """Batch of text strings -> np.ndarray [N, D] L2-normalized (used for captions)"""
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = clip_processor(text=batch, return_tensors="pt", padding=True).to(device)
        text_outputs = clip_model.text_model(**inputs)
        pooled_output = text_outputs.pooler_output
        feats = clip_model.text_projection(pooled_output)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_embeds.append(feats.cpu().numpy())
    return np.concatenate(all_embeds, axis=0)

# --- Load SmolVLM2 (frozen) ---
smolvlm_name = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"
vlm_processor = AutoProcessor.from_pretrained(smolvlm_name)
vlm_model = AutoModelForImageTextToText.from_pretrained(
    smolvlm_name,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).to(device).eval()

@torch.no_grad()
def describe_image(image, prompt="Describe this image in one sentence."):
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
    ]
    text_prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vlm_processor(text=text_prompt, images=[image], return_tensors="pt").to(device)
    generated = vlm_model.generate(**inputs, max_new_tokens=64)
    return vlm_processor.batch_decode(generated, skip_special_tokens=True)[0]

print("Setup complete: dataset_a, images_a, clip_model, clip_processor, "
      "embed_images, embed_text, embed_text_query, embed_texts, "
      "vlm_model, vlm_processor, describe_image are all ready.")


In [ ]:
# %% CELL 3.1 — Suppress the harmless processor warning noise
import warnings
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*processor_kwargs.*")


## 1. Encode the archive and build an exact index

CLIP is frozen throughout. The `get_image_features` / `get_text_features` wrappers are bypassed deliberately: in the pinned `transformers` version they return a pooled output rather than the projected embedding.


In [ ]:
# %% CELL 5 — Extract embeddings for Dataset A
images_a = [dataset_a[i]["image"].convert("RGB") for i in range(len(dataset_a))]

t0 = time.time()
embeddings_a = embed_images(images_a)
t1 = time.time()
print(f"Embedded {len(images_a)} images in {t1 - t0:.2f}s "
      f"({(t1 - t0) / len(images_a) * 1000:.2f} ms/image)")
print("Embedding shape:", embeddings_a.shape)

np.save("artifacts/embeddings_a.npy", embeddings_a)  # save so you don't recompute every session


In [ ]:
# %% CELL 6 — Build FAISS index (flat, exact search — baseline for later comparisons)
d = embeddings_a.shape[1]
index_flat = faiss.IndexFlatIP(d)  # inner product on L2-normalized vectors = cosine similarity
index_flat.add(embeddings_a)
print(f"FAISS index built: {index_flat.ntotal} vectors, dim={d}")

faiss.write_index(index_flat, "artifacts/index_a_flat.index")


In [ ]:
# %% CELL 7 — Sanity check: text-to-image retrieval

@torch.no_grad()
def embed_text(query):
    inputs = clip_processor(text=[query], return_tensors="pt", padding=True).to(device)
    text_outputs = clip_model.text_model(**inputs)
    pooled_output = text_outputs.pooler_output
    feats = clip_model.text_projection(pooled_output)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()

def retrieve_topk(query_text, k=5):
    q_embed = embed_text(query_text)
    scores, indices = index_flat.search(q_embed, k)
    return scores[0], indices[0]

test_queries = ["a dog running on grass", "people at a table eating", "a red car on the street"]
for q in test_queries:
    scores, idx = retrieve_topk(q, k=5)
    print(f"\nQuery: '{q}'")
    for rank, (s, i) in enumerate(zip(scores, idx)):
        caption = dataset_a[int(i)]["caption"][0] if "caption" in dataset_a.column_names else "(no caption field)"
        print(f"  #{rank+1} score={s:.3f} idx={i} caption={caption}")

# Eyeball these results — if retrieved images/captions look semantically related to the
# query, CLIP + FAISS retrieval is working correctly. This is your Day 1 checkpoint.
 


## 2. The gate: retrieval-gated pipeline vs VLM-on-everything

The baseline is timed over a 50-image subset and extrapolated — running the VLM over all 2,000 images would take over an hour, which is the point the paper is making.


In [ ]:
@torch.no_grad()
def embed_text_query(query):
    inputs = clip_processor(text=[query], return_tensors="pt", padding=True).to(
        clip_model.device
    )
    text_outputs = clip_model.text_model(**inputs)
    pooled_output = text_outputs.pooler_output
    feats = clip_model.text_projection(pooled_output)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()

def retrieval_gated_pipeline(query_text, images_pool, index, k=5, vlm_prompt=None):
    prompt = vlm_prompt or f"Does this image show: {query_text}? Answer in one sentence."
    q_embed = embed_text_query(query_text)
    scores, indices = index.search(q_embed, k)
    results = []
    for idx in indices[0]:
        img = images_pool[int(idx)]
        output = describe_image(img, prompt=prompt)
        results.append((int(idx), output))
    return results


In [ ]:
# %% CELL 4 — HEADLINE COMPARISON: retrieval-gated vs VLM-on-everything
# This produces the core efficiency number for the paper.


query = "a dog running on grass"
K = 5
N_BASELINE = 50  # run VLM-on-everything over a subset (not all 2000 — too slow) for a fair per-image comparison

# --- Retrieval-gated ---
t0 = time.time()
gated_results = retrieval_gated_pipeline(query, images_a, index_flat, k=K)
t1 = time.time()
gated_time = t1 - t0
print(f"Retrieval-gated pipeline: {gated_time:.2f}s for k={K} images "
      f"({gated_time / K:.2f} s/image, plus retrieval overhead)")

# --- VLM-on-everything baseline (subset, for fair per-image timing) ---
t0 = time.time()
for img in images_a[:N_BASELINE]:
    _ = describe_image(img, prompt=f"Does this image show: {query}? Answer in one sentence.")
t1 = time.time()
baseline_time = t1 - t0
print(f"VLM-on-everything baseline: {baseline_time:.2f}s for {N_BASELINE} images "
      f"({baseline_time / N_BASELINE:.2f} s/image)")

# --- Extrapolated cost comparison ---
per_image_vlm_cost = baseline_time / N_BASELINE
full_pool_cost_no_retrieval = per_image_vlm_cost * len(images_a)
full_pool_cost_with_retrieval = per_image_vlm_cost * K  # only K images hit the VLM

print(f"\nExtrapolated VLM cost over full pool ({len(images_a)} images):")
print(f"  Without retrieval gating: {full_pool_cost_no_retrieval:.1f}s")
print(f"  With retrieval gating (k={K}): {full_pool_cost_with_retrieval:.1f}s")
print(f"  Speedup factor: {full_pool_cost_no_retrieval / full_pool_cost_with_retrieval:.1f}x")

# This speedup number (driven by len(images_a)/k) is expected to be large and somewhat
# trivial on its own — the paper's real contribution is showing accuracy/recall is
# preserved despite only inspecting k candidates. Log gated_results and eyeball them
# for correctness before writing this up.
 


## 3. Linear adapter on Flickr30k

0.26M parameters (512x512 + bias), identity-initialised, trained with symmetric InfoNCE against caption embeddings. CLIP stays frozen.


In [ ]:
# %% CELL 2 — Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


In [ ]:
# %% CELL 3 — Define the adapter (identity-initialized linear projection)
class LinearAdapter(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.proj = nn.Linear(embed_dim, embed_dim)
        nn.init.eye_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        out = self.proj(x)
        return out / out.norm(dim=-1, keepdim=True)

embed_dim = embeddings_a.shape[1]
adapter = LinearAdapter(embed_dim).to(device)
optimizer = torch.optim.Adam(adapter.parameters(), lr=1e-4)


In [ ]:
# %% CELL 4 — Embed Dataset A captions (uses the fixed embed_texts from the master cell)
text_embeddings_a = embed_texts([dataset_a[i]["caption"][0] for i in range(len(dataset_a))])
print("Text embeddings shape:", text_embeddings_a.shape)

# Save immediately — Day 4 needs this file and it's easy to forget
np.save("artifacts/text_embeddings_a.npy", text_embeddings_a)

image_embeds_tensor = torch.tensor(embeddings_a, dtype=torch.float32)
text_embeds_tensor = torch.tensor(text_embeddings_a, dtype=torch.float32)


In [ ]:
# %% CELL 5 — Contrastive training loop
def contrastive_loss(image_embeds, text_embeds, temperature=0.07):
    logits = image_embeds @ text_embeds.T / temperature
    labels = torch.arange(logits.shape[0], device=logits.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2

EPOCHS = 5
BATCH_SIZE = 128
n_samples = image_embeds_tensor.shape[0]

adapter.train()
for epoch in range(EPOCHS):
    perm = torch.randperm(n_samples)
    total_loss = 0.0
    n_batches = 0
    for i in range(0, n_samples, BATCH_SIZE):
        batch_idx = perm[i:i + BATCH_SIZE]
        img_batch = image_embeds_tensor[batch_idx].to(device)
        txt_batch = text_embeds_tensor[batch_idx].to(device)

        adapted_img = adapter(img_batch)
        loss = contrastive_loss(adapted_img, txt_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    print(f"Epoch {epoch+1}/{EPOCHS}  avg loss: {total_loss / n_batches:.4f}")

torch.save(adapter.state_dict(), "artifacts/adapter_dataset_a.pt")
 


In [ ]:
# %% CELL 6 — Recall@k evaluation: adapter vs no-adapter
adapter.eval()

@torch.no_grad()
def recall_at_k(image_embeds, text_embeds, k_values=(1, 5, 10), use_adapter=False):
    img = image_embeds.to(device)
    txt = text_embeds.to(device)
    if use_adapter:
        img = adapter(img)

    sims = txt @ img.T
    ranks = []
    for i in range(sims.shape[0]):
        sorted_idx = torch.argsort(sims[i], descending=True)
        rank = (sorted_idx == i).nonzero(as_tuple=True)[0].item()
        ranks.append(rank)
    ranks = np.array(ranks)

    return {f"R@{k}": float((ranks < k).mean()) for k in k_values}

no_adapter_results = recall_at_k(image_embeds_tensor, text_embeds_tensor, use_adapter=False)
with_adapter_results = recall_at_k(image_embeds_tensor, text_embeds_tensor, use_adapter=True)

print("Without adapter (raw CLIP space):", no_adapter_results)
print("With adapter (trained on Dataset A):", with_adapter_results)


In [ ]:
# %% CELL 7 — Save results for the paper / abstract
import json
day3_results = {
    "no_adapter": no_adapter_results,
    "with_adapter": with_adapter_results,
}
with open("artifacts/day3_recall_results.json", "w") as f:
    json.dump(day3_results, f, indent=2)

print("\nDay 3 done when: adapter_dataset_a.pt, text_embeddings_a.npy, and "
      "day3_recall_results.json all exist. This R@5 number (with_adapter) is "
      "the recall figure to pair with Day 2's 400x speedup in the abstract.")
 


## 4. Dataset B (BDD100K) and the hybrid adapter

BDD100K scenario-classification ships without natural-language captions, so captions here are pseudo-labels generated by the same frozen SmolVLM2 used downstream. This is a stated limitation, not an oversight: the BDD100K recall numbers partly measure self-consistency.


In [ ]:
# Point this at your local copy of the BDD100K scenario-classification val split.
# Note the space in the folder name.
DATASET_B_PATH = os.environ.get(
    'DATASET_B_PATH',
    '/kaggle/input/datasets/marquis03/bdd100k-scenario-classification/val/city street',
)


In [ ]:
# %% CELL 5 — Load Dataset B (static BDD100K scenario images — confirmed working)
import os
from PIL import Image

def load_dataset_b_from_folder(folder_path, max_images=500):
    valid_ext = (".jpg", ".jpeg", ".png")
    files = sorted(f for f in os.listdir(folder_path) if f.lower().endswith(valid_ext))[:max_images]
    images_b = [Image.open(os.path.join(folder_path, f)).convert("RGB") for f in files]
    return images_b, files


images_b, filenames_b = load_dataset_b_from_folder(DATASET_B_PATH, max_images=500)
print(f"Loaded {len(images_b)} Dataset B images (city street scenes)")
print(f"Example filename: {filenames_b[0]}")


In [ ]:
# %% CELL 6 — Embed Dataset B: generate pseudo-captions with frozen SmolVLM2, then embed
# BDD100K scenario-classification has no natural-language captions — this is a stated
# limitation to flag explicitly in the paper (pseudo-labels, not ground truth).

captions_b = []
for i, img in enumerate(images_b):
    caption = describe_image(img, prompt="Describe this driving scene in one sentence.")
    captions_b.append(caption)
    if i < 3:
        print(f"Sample caption {i}: {caption}")

print(f"\nGenerated {len(captions_b)} pseudo-captions for Dataset B")

# --- Clean captions: strip the echoed chat-template prompt, keep only the response ---
def clean_caption(raw_output):
    """Extract just the assistant's response from SmolVLM2's full chat-template output."""
    if "Assistant:" in raw_output:
        return raw_output.split("Assistant:")[-1].strip()
    return raw_output.strip()

captions_b_clean = [clean_caption(c) for c in captions_b]
print("Cleaned sample:", captions_b_clean[0])

# --- Re-define embed_texts with truncation, to guard against CLIP's 77-token limit ---
@torch.no_grad()
def embed_texts(texts, batch_size=64):
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = clip_processor(
            text=batch, return_tensors="pt", padding=True,
            truncation=True, max_length=77,   # CLIP's hard limit
        ).to(device)
        text_outputs = clip_model.text_model(**inputs)
        pooled_output = text_outputs.pooler_output
        feats = clip_model.text_projection(pooled_output)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_embeds.append(feats.cpu().numpy())
    return np.concatenate(all_embeds, axis=0)

embeddings_b = embed_images(images_b)
text_embeddings_b = embed_texts(captions_b_clean)

np.save("artifacts/embeddings_b.npy", embeddings_b)
np.save("artifacts/text_embeddings_b.npy", text_embeddings_b)
print(f"Dataset B embeddings saved: {embeddings_b.shape[0]} images")


In [ ]:
# %% CELL 7 — Build hybrid training set + train adapter
image_embeds_a = torch.tensor(np.load("artifacts/embeddings_a.npy"), dtype=torch.float32)
text_embeds_a = torch.tensor(np.load("artifacts/text_embeddings_a.npy"), dtype=torch.float32)
image_embeds_b = torch.tensor(embeddings_b, dtype=torch.float32)
text_embeds_b = torch.tensor(text_embeddings_b, dtype=torch.float32)

image_embeds_hybrid = torch.cat([image_embeds_a, image_embeds_b], dim=0)
text_embeds_hybrid = torch.cat([text_embeds_a, text_embeds_b], dim=0)
origin_labels = np.array(["A"] * len(image_embeds_a) + ["B"] * len(image_embeds_b))

print(f"Hybrid training set: {len(image_embeds_hybrid)} pairs "
      f"({len(image_embeds_a)} from A, {len(image_embeds_b)} from B)")

embed_dim = image_embeds_hybrid.shape[1]
adapter_hybrid = LinearAdapter(embed_dim).to(device)
optimizer = torch.optim.Adam(adapter_hybrid.parameters(), lr=1e-4)

EPOCHS = 8
BATCH_SIZE = 128
n_samples = image_embeds_hybrid.shape[0]

adapter_hybrid.train()
for epoch in range(EPOCHS):
    perm = torch.randperm(n_samples)
    total_loss, n_batches = 0.0, 0
    for i in range(0, n_samples, BATCH_SIZE):
        batch_idx = perm[i:i + BATCH_SIZE]
        img_batch = image_embeds_hybrid[batch_idx].to(device)
        txt_batch = text_embeds_hybrid[batch_idx].to(device)
        adapted_img = adapter_hybrid(img_batch)
        loss = contrastive_loss(adapted_img, txt_batch)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item(); n_batches += 1
    print(f"Epoch {epoch+1}/{EPOCHS}  avg loss: {total_loss / n_batches:.4f}")

torch.save(adapter_hybrid.state_dict(), "artifacts/adapter_hybrid.pt")

# %% CELL 8 — Split evaluation (the generalization result)
adapter_hybrid.eval()

@torch.no_grad()
def recall_at_k(image_embeds, text_embeds, adapter=None, k_values=(1, 5, 10)):
    img = image_embeds.to(device)
    txt = text_embeds.to(device)
    if adapter is not None:
        img = adapter(img)
    sims = txt @ img.T
    ranks = []
    for i in range(sims.shape[0]):
        sorted_idx = torch.argsort(sims[i], descending=True)
        ranks.append((sorted_idx == i).nonzero(as_tuple=True)[0].item())
    ranks = np.array(ranks)
    return {f"R@{k}": float((ranks < k).mean()) for k in k_values}

mask_a = origin_labels == "A"
mask_b = origin_labels == "B"

results_a_noadapter = recall_at_k(image_embeds_hybrid[mask_a], text_embeds_hybrid[mask_a], adapter=None)
results_a_hybrid = recall_at_k(image_embeds_hybrid[mask_a], text_embeds_hybrid[mask_a], adapter=adapter_hybrid)
results_b_noadapter = recall_at_k(image_embeds_hybrid[mask_b], text_embeds_hybrid[mask_b], adapter=None)
results_b_hybrid = recall_at_k(image_embeds_hybrid[mask_b], text_embeds_hybrid[mask_b], adapter=adapter_hybrid)

print("Dataset A — no adapter:     ", results_a_noadapter)
print("Dataset A — hybrid adapter: ", results_a_hybrid)
print("Dataset B — no adapter:     ", results_b_noadapter)
print("Dataset B — hybrid adapter: ", results_b_hybrid)

all_results = {
    "A_no_adapter": results_a_noadapter, "A_hybrid_adapter": results_a_hybrid,
    "B_no_adapter": results_b_noadapter, "B_hybrid_adapter": results_b_hybrid,
}
with open("artifacts/day4_generalization_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("\nDay 4 complete: hybrid adapter trained across Flickr30k + BDD100K city-street scenes.")


## 5. Index comparison: Flat vs IVF vs HNSW

Combined pool of 2,500 vectors. Flat is the ground truth for the recall proxy; the sweep varies `nprobe` (IVF) and `efSearch` (HNSW).


In [ ]:
# %% CELL 2 — Imports + suppress warning noise
import warnings
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*processor_kwargs.*")

import faiss
import numpy as np
import time
import json


In [ ]:
# %% CELL 3 — Build the combined pool (reuses Day 4's embeddings if in memory,
# falls back to the raw embeddings_a/embeddings_b arrays otherwise)
try:
    embeddings_all = np.concatenate(
        [image_embeds_hybrid.numpy() if hasattr(image_embeds_hybrid, "numpy") else image_embeds_hybrid],
        axis=0,
    ).astype("float32")
except NameError:
    embeddings_all = np.concatenate([embeddings_a, embeddings_b], axis=0).astype("float32")

d = embeddings_all.shape[1]
n = embeddings_all.shape[0]
print(f"Combined pool: {n} vectors, dim: {d} (2000 from A + 500 from B)")
 


In [ ]:
# %% CELL 4 — Build each index type, timing the build
index_configs = {}

# --- Flat (exact search, ground-truth baseline) ---
t0 = time.time()
index_flat = faiss.IndexFlatIP(d)
index_flat.add(embeddings_all)
t1 = time.time()
index_configs["flat"] = {"index": index_flat, "build_time": t1 - t0}

# --- IVF (inverted file, approximate, needs training) ---
nlist = min(100, n // 39)  # FAISS wants ~40+ points per cluster minimum
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
t0 = time.time()
index_ivf.train(embeddings_all)
index_ivf.add(embeddings_all)
t1 = time.time()
index_ivf.nprobe = 10
index_configs["ivf"] = {"index": index_ivf, "build_time": t1 - t0}

# --- HNSW (graph-based, approximate, no training needed) ---
M = 32
index_hnsw = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)
t0 = time.time()
index_hnsw.add(embeddings_all)
t1 = time.time()
index_configs["hnsw"] = {"index": index_hnsw, "build_time": t1 - t0}

for name, cfg in index_configs.items():
    print(f"{name}: build_time={cfg['build_time']:.4f}s, ntotal={cfg['index'].ntotal}, nlist={nlist if name=='ivf' else '-'}")

# %% CELL 5 — Query latency + recall@k for each index type (vs flat as ground truth)
def get_ground_truth(index_flat, queries, k=10):
    _, indices = index_flat.search(queries, k)
    return indices

def evaluate_index(index, queries, ground_truth, k=10, n_repeats=5):
    times = []
    for _ in range(n_repeats):
        t0 = time.time()
        _, indices = index.search(queries, k)
        t1 = time.time()
        times.append(t1 - t0)
    avg_latency = np.mean(times) / len(queries)
    std_latency = np.std(times) / len(queries)

    recalls = []
    for i in range(len(queries)):
        retrieved = set(indices[i].tolist())
        true = set(ground_truth[i].tolist())
        recalls.append(len(retrieved & true) / k)
    avg_recall = np.mean(recalls)

    return {
        "avg_query_latency_ms": avg_latency * 1000,
        "std_query_latency_ms": std_latency * 1000,
        "recall@k_vs_flat": avg_recall,
    }

n_queries = min(200, n // 5)  # scale queries sensibly with pool size
rng = np.random.default_rng(42)
query_idx = rng.choice(n, size=n_queries, replace=False)
queries = embeddings_all[query_idx]

k = 10
ground_truth = get_ground_truth(index_configs["flat"]["index"], queries, k=k)

index_results = {}
for name, cfg in index_configs.items():
    result = evaluate_index(cfg["index"], queries, ground_truth, k=k)
    result["build_time_s"] = cfg["build_time"]
    index_results[name] = result
    print(f"\n{name}:")
    print(f"  build_time: {result['build_time_s']:.4f}s")
    print(f"  avg_query_latency: {result['avg_query_latency_ms']:.4f} ± {result['std_query_latency_ms']:.4f} ms")
    print(f"  recall@{k} vs flat: {result['recall@k_vs_flat']:.3f}")


In [ ]:
# %% CELL 6 — nprobe / ef_search sweep — the latency-recall tradeoff curve
nprobe_values = [1, 5, 10, 20, 50]
ivf_sweep = []
for nprobe in nprobe_values:
    index_configs["ivf"]["index"].nprobe = nprobe
    result = evaluate_index(index_configs["ivf"]["index"], queries, ground_truth, k=k)
    result["nprobe"] = nprobe
    ivf_sweep.append(result)
    print(f"IVF nprobe={nprobe}: latency={result['avg_query_latency_ms']:.4f}ms "
          f"recall={result['recall@k_vs_flat']:.3f}")

ef_search_values = [16, 32, 64, 128, 256]
hnsw_sweep = []
for ef in ef_search_values:
    index_configs["hnsw"]["index"].hnsw.efSearch = ef
    result = evaluate_index(index_configs["hnsw"]["index"], queries, ground_truth, k=k)
    result["ef_search"] = ef
    hnsw_sweep.append(result)
    print(f"HNSW ef_search={ef}: latency={result['avg_query_latency_ms']:.4f}ms "
          f"recall={result['recall@k_vs_flat']:.3f}")
 


In [ ]:
# %% CELL 7 — Save results
with open("artifacts/day5_index_comparison.json", "w") as f:
    json.dump({
        "pool_size": n,
        "pool_composition": {"dataset_a": 2000, "dataset_b": 500},
        "base_comparison": index_results,
        "ivf_sweep": ivf_sweep,
        "hnsw_sweep": hnsw_sweep,
    }, f, indent=2)


In [ ]:
# %% CELL 8 — Latency-recall tradeoff plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot([r["avg_query_latency_ms"] for r in ivf_sweep],
        [r["recall@k_vs_flat"] for r in ivf_sweep], marker="o", label="IVF (varying nprobe)")
ax.plot([r["avg_query_latency_ms"] for r in hnsw_sweep],
        [r["recall@k_vs_flat"] for r in hnsw_sweep], marker="s", label="HNSW (varying ef_search)")
ax.set_xlabel("Query latency (ms)")
ax.set_ylabel(f"Recall@{k} vs exact search")
ax.set_title(f"Index latency-recall tradeoff (n={n} vectors, A+B combined)")
ax.legend()
plt.tight_layout()
plt.savefig("artifacts/latency_recall_tradeoff.png", dpi=150)
plt.show()

print(f"\nDay 5 done: day5_index_comparison.json + latency_recall_tradeoff.png saved. "
      f"Pool scale: {n} vectors (2000 Flickr30k + 500 BDD100K).")


## 6. Quantization ablation: FP16 vs INT8 vs NF4

This is the counterintuitive result. Batch size is one and only one model is resident at a time — the regime an on-device deployment actually sits in, and the regime INT8's mixed-precision decomposition overhead cannot be amortised in.


In [ ]:
!pip install -q -U bitsandbytes


In [ ]:
import warnings
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*processor_kwargs.*")

import torch
import time
import json
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Quantized inference needs a GPU runtime — check Kaggle accelerator settings."

smolvlm_name = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"
# vlm_processor should already be in memory from the master cell — reload if fresh session:
# vlm_processor = AutoProcessor.from_pretrained(smolvlm_name)


In [ ]:
# %% CELL 3 — Load three precision variants of the same model
model_fp16 = AutoModelForImageTextToText.from_pretrained(
    smolvlm_name, torch_dtype=torch.float16
).to(device).eval()

bnb_int8 = BitsAndBytesConfig(load_in_8bit=True)
model_int8 = AutoModelForImageTextToText.from_pretrained(
    smolvlm_name, quantization_config=bnb_int8, device_map="auto"
).eval()

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model_4bit = AutoModelForImageTextToText.from_pretrained(
    smolvlm_name, quantization_config=bnb_4bit, device_map="auto"
).eval()

models = {"fp16": model_fp16, "int8": model_int8, "4bit": model_4bit}


In [ ]:
# %% CELL 4 — Memory footprint per variant
def get_model_memory_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)

memory_results = {name: get_model_memory_mb(m) for name, m in models.items()}
for name, mem in memory_results.items():
    print(f"{name}: {mem:.1f} MB")
 


In [ ]:
# %% CELL 5 — Inference helper (works across all three precisions)
@torch.no_grad()
def run_inference(model, image, prompt="Describe this image in one sentence."):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text_prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vlm_processor(text=text_prompt, images=[image], return_tensors="pt").to(model.device)
    generated = model.generate(**inputs, max_new_tokens=64)
    return vlm_processor.batch_decode(generated, skip_special_tokens=True)[0]

def clean_output(raw_output):
    """Strip the echoed chat-template prompt, keep only the model's response."""
    if "Assistant:" in raw_output:
        return raw_output.split("Assistant:")[-1].strip()
    return raw_output.strip()
 


In [ ]:
# %% CELL 6 — Latency comparison across precisions (quieter output)
import warnings
import logging
import transformers

# Broader suppression — catches processor_kwargs and any other transformers-level noise
transformers.logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

sample_images = images_a[:20]
latency_results = {}

for name, model in models.items():
    print(f"Running {name}...")
    _ = run_inference(model, sample_images[0])  # warmup

    t0 = time.time()
    outputs = []
    for i, img in enumerate(sample_images):
        outputs.append(run_inference(model, img))
        if (i + 1) % 5 == 0:
            print(f"  {name}: {i + 1}/{len(sample_images)} images done")
    t1 = time.time()

    avg_latency = (t1 - t0) / len(sample_images)
    latency_results[name] = {
        "avg_latency_s": avg_latency,
        "sample_outputs": [clean_output(o) for o in outputs[:3]],
    }
    print(f"{name}: {avg_latency:.3f} s/image\n")


In [ ]:
# %% CELL 7 — Qualitative side-by-side comparison
print("\n--- Qualitative comparison (first 3 images) ---")
for i in range(3):
    print(f"\nImage {i}:")
    for name in models:
        print(f"  [{name}] {latency_results[name]['sample_outputs'][i]}")


In [ ]:
# Save the quantization results
import json

fp16_mem = memory_results["fp16"]
fp16_lat = latency_results["fp16"]["avg_latency_s"]

day6_results = {
    "memory_mb": memory_results,
    "latency_s_per_image": {k: v["avg_latency_s"] for k, v in latency_results.items()},
    "memory_savings_vs_fp16_pct": {
        k: (fp16_mem - v) / fp16_mem * 100 for k, v in memory_results.items()
    },
    "latency_ratio_vs_fp16": {
        k: v["avg_latency_s"] / fp16_lat for k, v in latency_results.items()
    },
}
with open("artifacts/day6_quantization_results.json", "w") as f:
    json.dump(day6_results, f, indent=2)
print(json.dumps(day6_results, indent=2))


## 7. Index maintenance and provenance logging

Offline archives are append-heavy and query-light, so incremental insert cost matters as much as query cost. The provenance record is emitted by the gate itself: because the gate already makes an explicit ranking decision before the VLM runs, logging *why* a candidate was selected separately from *what* the model concluded costs nothing measurable against a 1.4 s VLM call.

This section was reconstructed from `results/day9_*.json` — see `src/provenance.py`.


In [ ]:
# %% Incremental insert vs full rebuild, per index type
import time
import numpy as np
import faiss

def time_incremental_inserts(build_fn, base_vectors, n_batches=10, batch_size=10, seed=0):
    """Mean/std wall time for appending one batch of vectors to a live index."""
    rng = np.random.default_rng(seed)
    d = base_vectors.shape[1]
    index = build_fn(d, base_vectors)
    per_batch_ms = []
    for _ in range(n_batches):
        new = rng.normal(size=(batch_size, d)).astype("float32")
        new /= np.linalg.norm(new, axis=1, keepdims=True)
        t0 = time.perf_counter()
        index.add(new)
        per_batch_ms.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(per_batch_ms)), float(np.std(per_batch_ms))


def time_full_rebuild(build_fn, base_vectors, n_repeats=5):
    d = base_vectors.shape[1]
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        build_fn(d, base_vectors)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


def build_flat(d, vecs):
    idx = faiss.IndexFlatIP(d)
    idx.add(vecs)
    return idx

def build_ivf(d, vecs, nlist=64):
    quantizer = faiss.IndexFlatIP(d)
    idx = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
    idx.train(vecs)          # incremental adds below reuse these centroids, no retraining
    idx.add(vecs)
    idx.nprobe = 10
    return idx

def build_hnsw(d, vecs, M=32):
    idx = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)
    idx.add(vecs)
    return idx


builders = {"flat": build_flat, "ivf": build_ivf, "hnsw": build_hnsw}
maintenance = {}
for name, fn in builders.items():
    mean_ms, std_ms = time_incremental_inserts(fn, embeddings_all)
    maintenance[name] = {
        "incremental_mean_ms_per_batch": mean_ms,
        "incremental_std_ms_per_batch": std_ms,
        "full_rebuild_ms": time_full_rebuild(fn, embeddings_all),
    }
    print(f"{name}: incremental {mean_ms:.3f} ms/batch, rebuild {maintenance[name]['full_rebuild_ms']:.2f} ms")

with open("artifacts/day9_incremental_index_results.json", "w") as f:
    json.dump(maintenance, f, indent=2)


In [ ]:
# %% Provenance-emitting gated query
import uuid
from datetime import datetime, timezone

def gated_query_with_provenance(query_text, images_pool, index, index_name, k=5,
                                vlm_name="SmolVLM2-256M"):
    """Run one gated query and return (results, provenance_record).

    The record separates *why* each candidate reached the model (rank + similarity
    score from the gate) from *what* the model said about it.
    """
    t0 = time.perf_counter()
    q_embed = embed_text_query(query_text)
    scores, indices = index.search(q_embed, k)

    candidates = [
        {"image_idx": int(i), "similarity_score": float(s)}
        for s, i in zip(scores[0], indices[0])
    ]

    prompt = f"Does this image show: {query_text}? Answer yes or no."
    vlm_outputs = [
        clean_output(describe_image(images_pool[c["image_idx"]], prompt=prompt))
        for c in candidates
    ]

    record = {
        "trace_id": str(uuid.uuid4()),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "query": query_text,
        "index_config": {"name": index_name, "k": k},
        "retrieved_candidates": candidates,
        "vlm_model": vlm_name,
        "vlm_output": vlm_outputs,
        "pipeline_latency_s": time.perf_counter() - t0,
    }
    return list(zip(candidates, vlm_outputs)), record


provenance_log = []
for q in ["a dog running on grass", "people at a table eating", "a red car on the street"]:
    _, record = gated_query_with_provenance(q, images_a, index_flat, "flat", k=5)
    provenance_log.append(record)
    print(f"{q!r}: {record['vlm_output']}  ({record['pipeline_latency_s']:.2f}s)")

with open("artifacts/day9_provenance_log.json", "w") as f:
    json.dump(provenance_log, f, indent=2)


---

### What the provenance record buys you

The third query above is the interesting one. Retrieval returns five candidates ranked by
similarity, but the VLM affirms only one of them — so the gate's ranking and the model's
judgement disagree, and the record makes that disagreement inspectable without a second run
and without sending anything off the device.

Computing the joint metric this enables (rate at which affirmative VLM answers coincide with
a retrieval the gate got right, per domain) is flagged as future work in Appendix C of the
paper and is **not** computed here.
